# Station Stacking - KLAX

Wide HRRR/GFS same-day 11am notebook for `KLAX`.

This workflow tunes XGBoost, LightGBM, and CatBoost with Optuna/TPE against RMSE on two fixed validation folds: train 2021-2023 validate 2024, train 2022-2024 validate 2025. It then trains on 2021-2025, tests on 2026, adds a Ridge meta-model stack, and simulates deterministic 2-degree weather brackets without calling Polymarket.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

STATION_ID = "KLAX"
FAST_MODE = False
OPTUNA_TRIALS = 100
STACK_OPTUNA_TRIALS = 50
OPTUNA_VERBOSE = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
from src.calibration.station_stacking import (
    StationStackingConfig,
    missing_model_dependencies,
    run_station_year_split_experiment,
)


## Model Scores


In [3]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_verbose=OPTUNA_VERBOSE,
)
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-06 09:01:52,181] A new study created in memory with name: no-name-5a51bccb-cda6-41ef-bb88-af5ce00f0324
[I 2026-06-06 09:02:11,125] Trial 0 finished with value: 4.6304156181185725 and parameters: {'n_estimators': 799, 'learning_rate': 0.12369619597856178, 'max_depth': 6, 'min_child_weight': 2.385234757844707, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.6677615511747083}. Best is trial 0 with value: 4.6304156181185725.
[I 2026-06-06 09:04:19,177] Trial 1 finished with value: 4.5521280921744784 and parameters: {'n_estimators': 1440, 'learning_rate': 0.0032515743808034223, 'max_depth': 8, 'min_child_weight': 8.23143373099555, 'gamma': 1.0616955533913808, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.2922905212920093}. Best is trial 1 with value: 4.5521280921744784.
[I 2026-06-06 09:04:42,449] Tri

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,664,1.959539,4.488667
1,validation_2024_2025,lightgbm,664,2.005477,4.501617
2,validation_2024_2025,catboost,664,2.046371,4.524158
3,validation_2024_2025,hrrr_raw,664,2.250086,4.725899
4,validation_2024_2025,gfs_raw,664,3.666696,5.648337
5,test_2026,xgboost,137,2.907005,4.680427
6,test_2026,lightgbm,137,2.959384,4.797926
7,test_2026,catboost,137,3.086178,4.930954
8,test_2026,ridge_stack,137,2.902888,4.767033
9,test_2026,hrrr_raw,137,3.164470,4.926098


## 2026 Weather Brackets


In [4]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,137,2.907005,4.680427,36.49635
1,lightgbm,137,2.959384,4.797926,33.576642
2,catboost,137,3.086178,4.930954,28.467153
3,ridge_stack,137,2.902888,4.767033,32.846715
4,hrrr_raw,137,3.164470,4.926098,26.277372
5,gfs_raw,137,4.237015,5.039493,8.029197


In [ ]:
import pandas as pd


def adjacent_brackets(bracket):
    if pd.isna(bracket):
        return []
    text = str(bracket).strip()
    if not text or "-" not in text:
        return []
    try:
        lower = int(text.split("-", 1)[0])
    except ValueError:
        return []
    return [
        f"{lower - 2}-{lower - 1}",
        f"{lower}-{lower + 1}",
        f"{lower + 2}-{lower + 3}",
    ]


bracket_3way = result.bracket_predictions.copy()

valid = bracket_3way["actual_bracket"].notna() & bracket_3way["predicted_bracket"].astype(str).str.strip().ne("")
bracket_3way = bracket_3way.loc[valid].copy()
bracket_3way["picked_brackets"] = bracket_3way["predicted_bracket"].map(adjacent_brackets)
bracket_3way["three_bracket_hit"] = bracket_3way.apply(
    lambda row: row["actual_bracket"] in row["picked_brackets"],
    axis=1,
)

three_bracket_accuracy = (
    bracket_3way
    .groupby("method", as_index=False)
    .agg(
        count=("three_bracket_hit", "size"),
        exact_bracket_accuracy_pct=("bracket_hit", lambda x: x.mean() * 100),
        three_bracket_accuracy_pct=("three_bracket_hit", lambda x: x.mean() * 100),
    )
    .sort_values("three_bracket_accuracy_pct", ascending=False)
)

three_bracket_accuracy
